<a href="https://colab.research.google.com/github/Eng-Waheedullah-wazir/flyrank-waheed-ml-intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng-Waheedullah-wazir/flyrank-waheed-ml-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')
df.shape
df.shape                                  # confirm 30000 rows, 44 cols
df['content_id'].nunique()                # should equal len(df) if truly 1 row/page
df['client_id'].nunique()                 # should be 32

32

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The checks below verify the row grain, row/client counts, duplicate content IDs, missing values, and the 90-day/30-day window fields available in this starter dataset.

The starter CSV does not contain a daily date column. Therefore the exact calendar start/end dates of the trailing 90-day window cannot be reconstructed from this file alone. The contract uses the documented window definition rather than inventing calendar dates.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check the fields used in the contract and the label distribution.

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "scroll_rate",
]

label_source = "trend_direction"

context_cols = [
    "content_id",
    "client_id",
]

excluded_cols = [
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used",
]

print("Feature columns:", feature_cols)
print("\nLabel source:", label_source)
print("\nContext columns:", context_cols)
print("\nExcluded columns:", excluded_cols)

print("\nLabel distribution:")
print(
    df["trend_direction"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nMissingness for selected fields:")
print(
    df[
        feature_cols + ["trend_direction"]
    ].isna().mean().sort_values(ascending=False)
)


Feature columns: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'scroll_rate']

Label source: trend_direction

Context columns: ['content_id', 'client_id']

Excluded columns: ['trend_direction', 'trend_pct', 'provider_used', 'model_used']

Label distribution:
trend_direction
down      16262
flat       1152
new        2236
stable     5962
up         4388
Name: count, dtype: int64

Missingness for selected fields:
scroll_rate        0.004167
impressions_90d    0.000000
clicks_90d         0.000000
ctr                0.000000
avg_position       0.000000
trend_direction    0.000000
dtype: float64


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The checks below verify the row grain, row/client counts, duplicate content IDs, missing values, and the 90-day/30-day window fields available in this starter dataset.

The starter CSV does not contain a daily date column. Therefore the exact calendar start/end dates of the trailing 90-day window cannot be reconstructed from this file alone. The contract uses the documented window definition rather than inventing calendar dates.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Grain check
duplicate_content_ids = (
    df.groupby("content_id")
      .size()
      .reset_index(name="row_count")
      .query("row_count > 1")
)

print("Duplicate content_id groups:", len(duplicate_content_ids))

# Counts
print("\nTotal rows:", len(df))
print("Unique content items:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

# Missingness
check_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "scroll_rate",
    "trend_direction",
]

missing = (
    df[check_cols]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("\nMissing percentage:")
print(missing)

# Window-related checks
print("\n90-day activity checks:")
print("Rows with impressions_90d >= 1:",
      (df["impressions_90d"] >= 1).sum())

print("\n30-day comparison fields:")
print(
    df[
        [
            "impressions_last_30d",
            "impressions_prev_30d",
            "clicks_last_30d",
            "clicks_prev_30d",
            "sessions_last_30d",
            "sessions_prev_30d",
        ]
    ].isna().mean().mul(100).round(2)
)


Duplicate content_id groups: 0

Total rows: 30000
Unique content items: 30000
Unique clients: 32

Missing percentage:
scroll_rate        0.42
impressions_90d    0.00
clicks_90d         0.00
ctr                0.00
avg_position       0.00
trend_direction    0.00
dtype: float64

90-day activity checks:
Rows with impressions_90d >= 1: 30000

30-day comparison fields:
impressions_last_30d    0.0
impressions_prev_30d    0.0
clicks_last_30d         0.0
clicks_prev_30d         0.0
sessions_last_30d       0.0
sessions_prev_30d       0.0
dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This starter dataset supports a useful observed decline proxy, but it cannot establish future causality.

First, the history is summarized into a trailing 90-day window, so this CSV cannot show the full daily path of an individual page.

Second, the decline label is based on the current 30-day comparison (`trend_direction`), not a future 30-day outcome. Therefore a model using this label is decision-support around the observed decline proxy, not a true future prediction.

Third, the GSC and GA4 measurements are not equally available for every page. Some fields are missing, and the missingness follows content type. A missing value therefore should not automatically be treated as zero.

Fourth, the 30-day comparison windows overlap the broader 90-day window. The trend fields must therefore stay out of the feature set because they contain the information used to define the label.

Finally, the starter CSV has no daily `report_date`, so exact calendar dates for the 90-day window cannot be reconstructed from this file alone.

In [11]:
# Contract limits: these are assertions, not a fourth data query.

assert len(df) == 30000
assert df["content_id"].nunique() == len(df)
assert df["client_id"].nunique() == 32

# The target source is kept separate from the feature list.
assert "trend_direction" not in feature_cols
assert "trend_pct" not in feature_cols
assert "content_id" not in feature_cols
assert "client_id" not in feature_cols

print("Contract checks passed.")
print("Features:", feature_cols)
print("Feature count:", len(feature_cols))

Contract checks passed.
Features: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'scroll_rate']
Feature count: 5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.